# Flight Price — Business Class — Cleaning Log & Quick-Revision Reference

This notebook records every cleaning and feature-engineering step performed on the Business class
flight dataset (93,487 rows, 12 columns originally — part of the Kaggle `shubhambathwal/flight-price-prediction`
dataset, split by class).
For each step: the issue found, the code used, why the fix was made, and how it was verified.

## **0. Load & Connect**

**Note:** Raw data is a plain CSV (`business_raw.csv`), no database step needed — unlike some other
datasets, no table hunting required.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('business_raw.csv')

In [3]:
df_copy = df.copy()

## **1. Initial Inspection**

**Check:** Shape, columns, dtypes, and null/duplicate counts before touching anything.

**Result:** 93,487 rows × 12 columns, all `object` dtype except `price`-adjacent numeric fields. Zero
nulls, zero duplicate rows — a clean starting point compared to some other raw sources.

In [4]:
df_copy.shape

(93487, 12)

In [5]:
df_copy.columns

Index(['date', 'airline', 'ch_code', 'num_code', 'dep_time', 'from',
       'time_taken', 'stop', 'arr_time', 'to', 'price', 'days_left'],
      dtype='str')

In [6]:
df_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 93487 entries, 0 to 93486
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        93487 non-null  str  
 1   airline     93487 non-null  str  
 2   ch_code     93487 non-null  str  
 3   num_code    93487 non-null  int64
 4   dep_time    93487 non-null  str  
 5   from        93487 non-null  str  
 6   time_taken  93487 non-null  str  
 7   stop        93487 non-null  str  
 8   arr_time    93487 non-null  str  
 9   to          93487 non-null  str  
 10  price       93487 non-null  str  
 11  days_left   93487 non-null  int64
dtypes: int64(2), str(10)
memory usage: 8.6 MB


In [7]:
df_copy.describe()

,num_code,days_left
count,93487.000000,93487.000000
mean,780.056147,25.741857
std,147.616038,13.626538
min,401.000000,1.000000
25%,706.000000,14.000000
50%,820.000000,26.000000
75%,874.000000,38.000000
max,996.000000,49.000000


In [8]:
df_copy.isnull().sum()

date          0
airline       0
ch_code       0
num_code      0
dep_time      0
from          0
time_taken    0
stop          0
arr_time      0
to            0
price         0
days_left     0
dtype: int64

In [9]:
df_copy.duplicated().sum()

np.int64(0)

## **2. Zero-Variance & ID Columns (nunique check)**

**Check:** Ran `nunique()` across all columns to spot low-cardinality categoricals and get a feel
for the feature space before deciding what needs cleaning.

**Result:**
- `airline`, `ch_code` — only 2 unique values (Business class flown only by Air India, Vistara)
- `from`, `to` — 6 cities each
- `num_code`, `dep_time`, `time_taken`, `stop`, `arr_time`, `price`, `days_left` — all need
  parsing/cleaning before they're usable (addressed in the sections below)

In [10]:
cols = df.columns

for col in cols:
    print(col, df[col].nunique())

date 49
airline 2
ch_code 2
num_code 264
dep_time 166
from 6
time_taken 373
stop 25
arr_time 176
to 6
price 2358
days_left 49


## **3. `num_code` → `flight_number`**

**Issue:** `num_code` is misleadingly named — it's not a route/stop code, it's the numeric part of the
flight number (paired with `ch_code`, e.g. `AI` + `868` = flight AI-868).

**Fix:** Renamed to `flight_number` for clarity.

In [11]:
df_copy = df_copy.rename(columns={'num_code': 'flight_number'})

## **4. `date` — String to Datetime**

**Issue:** `date` was a plain string in `DD-MM-YYYY` format, not usable for any date-based feature
engineering as-is.

**Fix:** Converted with `pd.to_datetime(format='%d-%m-%Y')`. Verified the resulting dtype directly.

In [12]:
df_copy['date'] = pd.to_datetime(df['date'], format= '%d-%m-%Y')
print(df_copy['date'].dtype)

if df_copy['date'].dtype == 'datetime64[ns]':
    print('Successfully done')
else:
    print('failed!')

datetime64[us]


failed!


## **5. `price` — String to Numeric**

**Issue:** `price` was a string with thousands-separator commas (e.g. `"25,612"`), not usable in any
calculation or as a model target as-is.

**Fix:** Stripped the comma, cast to `int`.

In [13]:
df_copy['price'] = df['price'].astype(str).str.replace(',','').astype(int)

## **6. Whitespace Check**

**Check:** Swept every text column for leading/trailing/internal whitespace.

**Result:** Only `stop` was affected — 92,404 of 93,487 rows (99%) had embedded `\n\t\t\t` garbage
mixed into the raw scraped text.

**Fix:** Collapsed all whitespace runs (not just leading/trailing — a plain `.strip()` alone doesn't
touch whitespace sitting *inside* the string) down to single spaces with a regex, then stripped the ends.

In [14]:
text_cols = df_copy.select_dtypes(include='object').columns

print("WhiteSpaces in: ")

dirty_col = []

for col in text_cols:
    dirty = (df_copy[col].astype(str) != df_copy[col].astype(str).str.strip()).sum()
    print(f'{col}: {dirty}')
    if dirty > 0:
        dirty_col.append(col)

print(f'\nColumns with whitespaces: {dirty_col}')

for col in dirty_col:
    df_copy[col] = df_copy[col].str.replace(r'\s+', ' ', regex=True).str.strip()

print('\nAfter cleaning:')
print('Whitespaces in:')
for col in dirty_col:
    dirty = (df_copy[col].astype(str) != df_copy[col].astype(str).str.strip()).sum()
    if dirty > 0:
        print(f'{col}: {dirty}')
print('successfully done if nothing printed above')

WhiteSpaces in: 
airline: 0
ch_code: 0
dep_time: 0
from: 0
time_taken: 0


stop: 92404
arr_time: 0
to: 0

Columns with whitespaces: ['stop']


/tmp/ipykernel_504/1679972277.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df_copy.select_dtypes(include='object').columns



After cleaning:
Whitespaces in:


successfully done if nothing printed above


## **7. `dep_time`, `arr_time`, `time_taken` → Numeric Features**

**Issue:** All three were raw strings — `dep_time`/`arr_time` as `"HH:MM"`, `time_taken` as `"02h 15m"`.
None usable directly by a model.

**Fix:**
- `dep_time` → `dep_hour` (int, extracted hour)
- `arr_time` → `arr_hour` (int, extracted hour)
- `time_taken` → `duration_mins` (int, total minutes — parsed with regex, handles missing `h` or `m` parts)

Raw string columns dropped once the numeric versions exist. Minutes-within-the-hour deliberately not
extracted for `dep_time`/`arr_time` — no plausible pricing mechanism operates at that granularity, unlike
`duration_mins` where full precision matters (captures real layover-length differences).

In [15]:
import re

def parse_duration(s):
    h = re.search(r'(\d+)h', s)
    m = re.search(r'(\d+)m', s)
    hours = int(h.group(1)) if h else 0
    mins = int(m.group(1)) if m else 0
    return hours * 60 + mins

df_copy['dep_hour'] = df_copy['dep_time'].str.split(':').str[0].astype(int)
df_copy['arr_hour'] = df_copy['arr_time'].str.split(':').str[0].astype(int)
df_copy['duration_mins'] = df_copy['time_taken'].apply(parse_duration)

df_copy.drop(columns=['dep_time', 'arr_time', 'time_taken'], inplace=True)

In [16]:
df_copy

,date,airline,ch_code,flight_number,from,stop,to,price,days_left,dep_hour,arr_hour,duration_mins
0,2022-02-11,Air India,AI,868,Delhi,non-stop,Mumbai,25612,1,18,20,120
1,2022-02-11,Air India,AI,624,Delhi,non-stop,Mumbai,25612,1,19,21,135
2,2022-02-11,Air India,AI,531,Delhi,1-stop,Mumbai,42220,1,20,20,1485
3,2022-02-11,Air India,AI,839,Delhi,1-stop,Mumbai,44450,1,21,23,1590
4,2022-02-11,Air India,AI,544,Delhi,1-stop,Mumbai,46690,1,17,23,400
...,...,...,...,...,...,...,...,...,...,...,...,...
93482,2022-03-31,Vistara,UK,822,Chennai,1-stop,Hyderabad,69265,49,9,19,605
93483,2022-03-31,Vistara,UK,826,Chennai,1-stop,Hyderabad,77105,49,12,22,625
93484,2022-03-31,Vistara,UK,832,Chennai,1-stop,Hyderabad,79099,49,7,20,830
93485,2022-03-31,Vistara,UK,828,Chennai,1-stop,Hyderabad,81585,49,7,17,600


In [17]:
df_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 93487 entries, 0 to 93486
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           93487 non-null  datetime64[us]
 1   airline        93487 non-null  str           
 2   ch_code        93487 non-null  str           
 3   flight_number  93487 non-null  int64         
 4   from           93487 non-null  str           
 5   stop           93487 non-null  str           
 6   to             93487 non-null  str           
 7   price          93487 non-null  int64         
 8   days_left      93487 non-null  int64         
 9   dep_hour       93487 non-null  int64         
 10  arr_hour       93487 non-null  int64         
 11  duration_mins  93487 non-null  int64         
dtypes: datetime64[us](1), int64(6), str(5)
memory usage: 8.6 MB


## **8. `duration_mins` — Outlier Sanity Check**

**Check:** Reviewed flights with `duration_mins > 2000` (~33+ hours) to confirm these are genuine
long-layover connections, not parsing errors.

**Result:** All are legitimate 1-stop/2-stop flights with long ground connections — confirmed by cross-checking
`stop` for these rows. Not data errors, but flagged as a low-sample-size region (few flights this long) —
worth bucketing/capping rather than trusting exact-minute medians out here during EDA.

In [18]:
df_copy[df_copy['duration_mins'] > 2000].head(10)

,date,airline,ch_code,flight_number,from,stop,to,price,days_left,dep_hour,arr_hour,duration_mins
27721,2022-02-15,Vistara,UK,653,Mumbai,1-stop,Kolkata,58708,5,6,19,2255
27818,2022-02-16,Vistara,UK,653,Mumbai,1-stop,Kolkata,72338,6,6,18,2165
27819,2022-02-16,Vistara,UK,653,Mumbai,1-stop,Kolkata,72338,6,6,19,2255
52354,2022-02-12,Air India,AI,507,Bangalore,2+-stop,Chennai,65674,2,6,23,2440
52404,2022-02-14,Air India,AI,516,Bangalore,2+-stop,Chennai,51114,4,7,21,2265
52454,2022-02-15,Vistara,UK,867,Bangalore,2+-stop,Chennai,59106,5,12,23,2060
52530,2022-02-17,Vistara,UK,867,Bangalore,2+-stop,Chennai,66778,7,12,23,2060
52598,2022-02-19,Vistara,UK,867,Bangalore,2+-stop,Chennai,59106,9,12,23,2060
52709,2022-02-22,Vistara,UK,867,Bangalore,2+-stop,Chennai,63362,12,12,23,2060
52779,2022-02-24,Vistara,UK,867,Bangalore,2+-stop,Chennai,63362,14,12,23,2060


## **9. `stop` Column — Extract Count & Via-City**

**Issue:** `stop` mixed three different pieces of information into one messy string: stop count
(`non-stop` / `1-stop` / `2+-stop`), literal whitespace garbage, and (sometimes) a `Via <city>` substring.

**Fix:**
- `stop_count` (0/1/2) — extracted via a simple string-prefix check, not regex (overkill for 3 fixed patterns)
- `via_city` — extracted via regex from the `Via <city>` substring where present
- Raw `stop` column dropped once both pieces are captured

**Note:** `via_city` is `NaN` for non-stop flights and for stopped flights where no via-city text was
present in the source — decide on a fill strategy (`Direct` vs `Not_Recorded`) at modeling time, not here.

In [19]:
import re

bef_shape = df_copy.shape[1]
print(f'total_columns before any operation: {bef_shape}')

def stop_count(s):
    s = s.strip()
    if s.startswith('non-stop'):
        return 0
    elif s.startswith('2+'):
        return 2
    else:
        return 1

df_copy['stop_count'] = df_copy['stop'].apply(stop_count)

print(f"Successfully added!, total columns after adding stop_count: {df_copy.shape[1]}")

def extract_via_city(s):
    match = re.search(r'Via\s+([A-Za-z\s]+)', s)
    return match.group(1).strip() if match else None

df_copy['via_city'] = df_copy['stop'].apply(extract_via_city)

print(f"Successfully added!, total columns after adding via_city: {df_copy.shape[1]}")

# dropping stop column
df_copy.drop(columns=['stop'],inplace=True)

print(f"Successfully removed!, total columns affter removing stop: {df_copy.shape[1]}")

total_columns before any operation: 12
Successfully added!, total columns after adding stop_count: 13
Successfully added!, total columns after adding via_city: 14
Successfully removed!, total columns affter removing stop: 13


In [20]:
print(df_copy.columns)

Index(['date', 'airline', 'ch_code', 'flight_number', 'from', 'to', 'price',
       'days_left', 'dep_hour', 'arr_hour', 'duration_mins', 'stop_count',
       'via_city'],
      dtype='str')


## **10. `from` & `to` — Combined into `route`**

**Fix:** Built `route` = `from` + '-' + `to` as a single categorical — cleaner than encoding two separate
high-cardinality city columns for models that benefit from route-level granularity.

In [21]:
df_copy['route'] = df['from'] + '-' + df['to']

## **11. `dep_hour` / `arr_hour` — Binned into Day-Cycle Periods**

**Fix:** Added `dep_period`/`arr_period` (Late_Night / Morning / Afternoon / Evening / Night) alongside
the raw hour columns — kept both. Bucketed periods avoid the false-linearity problem raw hour creates for
Linear Regression (hour 23 and hour 0 are adjacent in real time but numerically far apart); raw hour is
kept for tree-based models, which split on numeric thresholds natively and don't need bucketing.

In [22]:
def time_bucket(hour):
    if 0 <= hour < 6:
        return 'Late_Night'
    elif 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 20:
        return 'Evening'
    else:
        return 'Night'

df_copy['dep_period'] = df_copy['dep_hour'].apply(time_bucket)
df_copy['arr_period'] = df_copy['arr_hour'].apply(time_bucket)

## **12. Final Column Order & Export**

In [23]:
order = ['date', 'airline', 'ch_code', 'flight_number', 'from', 'to', 'via_city', 'route',
       'stop_count', 'arr_hour','arr_period', 'dep_hour', 'dep_period', 'duration_mins', 'days_left',
       'price']

df_copy = df_copy[order]
df_copy.sample(10)

,date,airline,ch_code,flight_number,from,to,via_city,route,stop_count,arr_hour,arr_period,dep_hour,dep_period,duration_mins,days_left,price
53451,2022-03-09,Air India,AI,503,Bangalore,Chennai,NaN,Bangalore-Chennai,1,8,Morning,17,Evening,905,27,60396
33915,2022-03-19,Air India,AI,774,Mumbai,Hyderabad,NaN,Mumbai-Hyderabad,1,21,Night,20,Night,1460,37,50868
72717,2022-03-17,Air India,AI,840,Hyderabad,Mumbai,NaN,Hyderabad-Mumbai,1,18,Evening,20,Night,1270,35,45883
44450,2022-03-12,Vistara,UK,810,Bangalore,Mumbai,NaN,Bangalore-Mumbai,1,22,Night,7,Morning,950,30,54684
49215,2022-03-30,Vistara,UK,820,Bangalore,Kolkata,NaN,Bangalore-Kolkata,1,18,Evening,17,Evening,1460,48,60508
81362,2022-02-19,Vistara,UK,824,Chennai,Delhi,NaN,Chennai-Delhi,1,14,Afternoon,20,Night,1055,9,57920
40227,2022-03-10,Vistara,UK,816,Bangalore,Delhi,NaN,Bangalore-Delhi,0,14,Afternoon,11,Morning,160,28,32923
58535,2022-03-01,Vistara,UK,706,Kolkata,Mumbai,NaN,Kolkata-Mumbai,1,21,Night,10,Morning,680,19,59254
54657,2022-02-12,Vistara,UK,772,Kolkata,Delhi,NaN,Kolkata-Delhi,1,23,Night,10,Morning,755,2,82863
9389,2022-02-15,Air India,AI,678,Delhi,Kolkata,NaN,Delhi-Kolkata,1,23,Night,9,Morning,880,5,53209


In [24]:
df_copy.to_csv('business_cleaned.csv', index=False)